# ReBRAC Stage D — `worldcomp-1000` critic-penalty-off mechanism probe

**目的**：验证 Phase 1 Step 2 暴露的 **Finding 1**——ReBRAC 在 worldcomp 上 dual penalty 几乎退化为单 penalty（`β2·critic_penalty_ratio = 0.0111`，比 crosscomp Stage C 低 1–2 个数量级）。

**机制猜想**：worldcomp 是 deterministic teacher policy，`next_actions = π_teacher(s')` 与 ReBRAC 自己的 `π_target(s')` 高度一致 → critic penalty `||π_target(s') + noise − a'||²` 退化为接近恒等约束。如果这个解释成立，那么把 β2 降到 0 应该几乎不影响最终成绩。

**核心问题**：`(β1=4.0, β2=0)` 在 `worldcomp-1000` deployable 上的 `mean_test_success` 是否落在 Phase 1 主 finalist `(β1=4.0, β2=2.0)` 的 `0.928 ± 0.077` 的 ±2pp 区间内？

**实验 scope**：

| 轴 | 配置 | 说明 |
|---|---|---|
| β1 | `4.0` | 与 Phase 1 主 finalist 一致 |
| β2 | **`0.0`** | 关键操纵：critic penalty off |
| dataset | `worldcomp-1000` | 与 Phase 1 同 dataset |
| seeds | `42 43` | 与 epoch-probe / Phase 1 baseline 重叠；**刻意不含 seed 44**——Phase 1 已暴露 seed 44 是 outlier，这里只验证 typical regime |
| TRAIN_EPOCHS | `64` | 与 Phase 1 一致 |
| 轨道 | deployable | 与 Phase 1 同协议 |
| val manifest | 40 episodes | 与 Phase 1 对齐（复用） |
| test manifest | 100 episodes | 与 Phase 1 对齐（复用） |

**预算**：2 seeds × 1 配置 × 64 epoch ≈ Phase 1 deployable 的 40% 算力。manifest 与离线数据集完全复用 Phase 1 已生成的资源。

**判据**：

| 情形 | mean_test_success | 解读 |
| --- | --- | --- |
| **A** 几乎一致 | `0.928 ± 2pp`（即 `0.908 ~ 0.948`） | Finding 1 机制猜想成立；Stage E ablation 中 critic-penalty-off 这一格直接收口 |
| **B** 略有下降 | `0.88 ~ 0.91` | dual penalty 在 worldcomp 上虽弱但仍非零；Stage E 仍需保留 critic-penalty-off 完整 5-seed ablation |
| **C** 显著下降 | `< 0.88` | Finding 1 机制猜想被证伪——critic penalty 在 worldcomp 上虽然 ratio 极小，但仍是关键稳定项；需要回查 Phase 1 `mean_critic_penalty_ratio` 的 per-seed 拆分 |

**输出树**（与 deployable / privileged_critic 完全隔离）：
- `checkpoints/offline/rebrac/worldcomp_critic_penalty_off_probe/`
- `results/offline/rebrac/worldcomp_critic_penalty_off_probe/`

## 0. 环境 sanity check

In [ ]:
!lscpu | head -10
print()
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

## 1. 环境配置

复用 driver `run_offline_rebrac_worldcomp_teacher_gap.sh` 的 deployable 通路，但把：
- β2 设为 0；
- 输出根目录隔离到 `worldcomp_critic_penalty_off_probe/`；
- seeds 收紧到 2（42/43）。

In [ ]:
import os

# —— 通用 ——
os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"

# —— 关键操纵：β2=0 ——
os.environ["ACTOR_PENALTY_COEF"]  = "4.0"
os.environ["CRITIC_PENALTY_COEF"] = "0.0"

# —— 协议（与 Phase 1 deployable 对齐）——
os.environ["DEPLOYABLE_FINAL_SEEDS"]                       = "42 43"
os.environ["DEPLOYABLE_FINAL_TRAIN_EPOCHS"]                = "64"
os.environ["DEPLOYABLE_FINAL_CHECKPOINT_EVERY_EPOCHS"]     = "8"
os.environ["FINAL_VAL_MANIFEST_EPISODES"]                  = "40"
os.environ["FINAL_TEST_MANIFEST_EPISODES"]                 = "100"

# —— 输出根目录隔离（不污染 Phase 1 / Phase 2）——
os.environ["DEPLOYABLE_RESULTS_ROOT"]    = "results/offline/rebrac/worldcomp_critic_penalty_off_probe"
os.environ["DEPLOYABLE_CHECKPOINT_ROOT"] = "checkpoints/offline/rebrac/worldcomp_critic_penalty_off_probe"

# manifest / 离线数据集复用 Phase 1（FINAL_MANIFEST_ROOT 用 driver 默认值）

## 2. 复用 sanity check

manifest 与离线数据集应已由 epoch-probe / Phase 1 生成；如果缺失，driver 会自动重建。

In [ ]:
import pathlib

manifest_val  = pathlib.Path("benchmarks/offline_rebrac_worldcomp_final/val_40/single_u10_cross_tgt15.json")
manifest_test = pathlib.Path("benchmarks/offline_rebrac_worldcomp_final/test_100/single_u10_cross_tgt15.json")
dataset       = pathlib.Path("offline_data/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz")

for label, p in [("val manifest", manifest_val), ("test manifest", manifest_test), ("offline dataset", dataset)]:
    if p.exists():
        print(f"[reuse] {label}: {p}")
    else:
        print(f"[warn] {label} 缺失：{p}（driver 会自动重建/重收）")

## 3. 全流程：deployable_train → validate → select → test → summarize

2 个 train run × 1 配置 × 64 epoch，每 8 epoch 一个 ckpt → val=40 选最佳 → test=100 出最终成绩。

In [ ]:
# 跑 critic-penalty-off probe 全流程
for mode in ["deployable_train", "deployable_validate", "deployable_test", "deployable_summarize"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh

## 4. 结果与 Phase 1 主 finalist 对照

In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS_ROOT = Path("results/offline/rebrac/worldcomp_critic_penalty_off_probe")
DATASET_NAME = "worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
PAIR_TAG     = "actorb_4p0__criticb_0p0"
SEEDS        = os.environ["DEPLOYABLE_FINAL_SEEDS"].split()


def load_test_per_seed() -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        path = RESULTS_ROOT / DATASET_NAME / PAIR_TAG / "test" / f"seed_{seed}.json"
        if not path.exists():
            print(f"[warn] missing: {path}")
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows.append({
            "seed": seed,
            "success_rate": payload["eval_success_rate"],
            "return": payload["eval_return"],
            "safety_cost": payload["eval_safety_cost"],
            "time_s": payload["eval_time_s"],
        })
    return pd.DataFrame(rows)


per_seed = load_test_per_seed()
print("[critic-penalty-off probe — per-seed test results]")
print(per_seed.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

if not per_seed.empty:
    print()
    probe_mean = per_seed["success_rate"].mean()
    probe_std  = per_seed["success_rate"].std() if len(per_seed) > 1 else 0.0
    probe_ret  = per_seed["return"].mean()

    # Phase 1 主 finalist 同 seeds (42/43) 的 test 成绩
    phase1_42 = 0.990
    phase1_43 = 0.930
    phase1_42_43_mean = (phase1_42 + phase1_43) / 2

    # Phase 1 5-seed mean (含 seed 44 outlier)
    phase1_5seed_mean = 0.928

    print("=" * 64)
    print("critic-penalty-off probe vs Phase 1 主 finalist 对照")
    print("-" * 64)
    print(f"{'config':<40}{'mean':>10}{'std':>10}")
    print("-" * 64)
    print(f"{'(β1=4.0, β2=0)  probe (2 seeds 42/43)':<40}{probe_mean:>10.4f}{probe_std:>10.4f}")
    print(f"{'(β1=4.0, β2=2)  Phase 1 (seeds 42/43)':<40}{phase1_42_43_mean:>10.4f}{'-':>10}")
    print(f"{'(β1=4.0, β2=2)  Phase 1 (5 seeds 42-46)':<40}{phase1_5seed_mean:>10.4f}{0.077:>10.4f}")
    print("=" * 64)
    print()

    delta_same_seeds = probe_mean - phase1_42_43_mean
    print(f"Δ vs Phase 1 同 seeds (42/43): {delta_same_seeds:+.4f} ({delta_same_seeds*100:+.1f}pp)")
    print()

    if abs(delta_same_seeds) <= 0.02:
        verdict = "情形 A：几乎一致 → Finding 1 机制猜想成立；Stage E critic-penalty-off 收口"
    elif probe_mean >= 0.88:
        verdict = "情形 B：略有下降 → dual penalty 在 worldcomp 上虽弱但非零；Stage E 仍需 5-seed ablation"
    else:
        verdict = "情形 C：显著下降 → Finding 1 机制猜想被证伪；需要 per-seed 复盘 Phase 1 critic_penalty_ratio"

    print(f"[probe 结论] {verdict}")

In [ ]:
# overview summary（critic_penalty / target_q）
import csv

overview_path = RESULTS_ROOT / "summaries" / "overview.csv"
if overview_path.exists():
    with overview_path.open(encoding="utf-8") as fp:
        reader = csv.DictReader(fp)
        for row in reader:
            print(f"[overview] dataset={row['dataset']} pair={row['pair']} num_seeds={row['num_seeds']}")
            print(f"  mean_test_success_rate    = {float(row['mean_test_success_rate']):.4f}")
            print(f"  std_test_success_rate     = {float(row['std_test_success_rate']):.4f}")
            print(f"  mean_test_return          = {float(row['mean_test_return']):.3f}")
            print(f"  mean_critic_penalty       = {float(row['mean_critic_penalty']):.4f}  (β2=0 时 critic loss 中无 penalty 项；这只是 logging 量)")
            print(f"  mean_target_q             = {float(row['mean_target_q']):.3f}")
            penalty_ratio = float(row['mean_critic_penalty_ratio'])
            print(f"  mean_critic_penalty_ratio = {penalty_ratio:.4f}  (β2=0 → β2·ratio = 0，但 ratio 本身仍可与 Phase 1 比较)")
else:
    print(f"[warn] overview.csv missing at {overview_path}")

## 5. 报告写入清单

跑完上面 cell 后：

1. 把 probe 数字 + 情形判定写入 [docs/rebrac_experiment_report.md](../docs/rebrac_experiment_report.md) §7.10.6 Finding 1 末尾（作为 "Finding 1 已被独立 probe 验证/证伪" 的补强）。
2. 在 [docs/rebrac_experiment_plan.md](../docs/rebrac_experiment_plan.md) §6.5.3 末尾的 "可选追加：critic-penalty-off probe" 段落标记为 `【已完成】`。
3. 根据情形：
   - **情形 A**：Stage E ablation 中 critic-penalty-off 这一格不再需要 5-seed 复跑，直接引用 probe 结果作为收口；
   - **情形 B**：Stage E 仍跑 5-seed `(β1=4.0, β2=0)` ablation；
   - **情形 C**：触发 Phase 1 per-seed `mean_critic_penalty_ratio` 复盘，重写 Finding 1 章节。